In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
import struct
from pathlib import Path
from scipy.stats import skew, kurtosis

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')


In [ ]:
class PipeDataset(Dataset):
    def __init__(self, data_list, labels, scaler=None, fit_scaler=False):
        self.data_list = data_list
        self.labels = labels
        if fit_scaler:
            all_data = np.vstack(data_list)
            self.scaler = StandardScaler().fit(all_data)
        else:
            self.scaler = scaler
        self.data_list = [torch.FloatTensor(self.scaler.transform(sample)) for sample in self.data_list]
        self.labels = torch.LongTensor(labels)
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        return self.data_list[idx], self.labels[idx]

def collate_fn(batch):
    data_list, labels = zip(*batch)
    max_len = max([d.shape[0] for d in data_list])
    padded_data, lengths = [], []
    for data in data_list:
        length = data.shape[0]
        lengths.append(length)
        padding = torch.zeros(max_len - length, data.shape[1])
        padded = torch.cat([data, padding], dim=0)
        padded_data.append(padded)
    data_batch = torch.stack(padded_data)
    labels_batch = torch.stack(labels)
    mask = torch.zeros(data_batch.shape[:2])
    for i, length in enumerate(lengths):
        mask[i, :length] = 1
    return data_batch, labels_batch, mask, torch.tensor(lengths)


In [ ]:
class PipeCNNEncoder(nn.Module):
    def __init__(self, input_channels=8, embedding_dim=64):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv1d(input_channels, 32, 9, padding=4), nn.BatchNorm1d(32), nn.ReLU())
        self.block = nn.Sequential(nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU())
        self.attention = nn.Sequential(nn.Conv1d(64, 32, 1), nn.ReLU(), nn.Conv1d(32, 1, 1))
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.stem(x)
        x = self.block(x)
        w = self.attention(x)
        w = torch.softmax(w, dim=2)
        x = (x * w).sum(dim=2)
        return x


In [ ]:
def read_raw_file(filepath):
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f'File not found: {path}')
    with open(path, 'rb') as f:
        data = f.read()
    if len(data) < 17:
        raise ValueError('File too short')
    x = struct.unpack_from('>i', data, 0)[0]
    y = struct.unpack_from('>i', data, 4)[0]
    thicknom = struct.unpack_from('>d', data, 8)[0]
    defective = struct.unpack_from('>B', data, 16)[0]
    num_values = x * y
    matrix = struct.unpack(f'>{num_values}d', data[17:17+num_values*8])
    matrix_np = np.array(matrix, dtype=np.float32).reshape(x, y)
    return {'matrix': matrix_np, 'defective': defective, 'thicknom': thicknom}

def load_dataset_from_folder(folder_path, verbose=True):
    folder = Path(folder_path)
    raw_files = list(folder.glob('*.raw'))
    if not raw_files:
        raise FileNotFoundError(f'No .raw files found in {folder_path}')
    if verbose:
        print(f'Found files: {len(raw_files)}')
    data_list, labels, thicknom_list = [], [], []
    for file_path in raw_files:
        try:
            result = read_raw_file(file_path)
            data_list.append(result['matrix'])
            labels.append(result['defective'])
            thicknom_list.append(result['thicknom'])
        except Exception as e:
            print(f'Error reading {file_path.name}: {e}')
    if verbose:
        print(f'Loaded: {len(data_list)} files')
    return data_list, np.array(labels), thicknom_list


In [ ]:
def extract_features(matrix, thicknom):
    m = matrix.copy()
    mask = m > 0
    if mask.any():
        rows = np.where(mask.any(axis=1))[0]
        m = m[rows[0]:rows[-1]+1]
    m = m / thicknom
    features = []
    flat = m.flatten()
    features += [flat.min(), flat.max(), flat.mean(), flat.std(), skew(flat), kurtosis(flat)]
    for col in range(m.shape[1]):
        col_data = m[:, col]
        features += [col_data.mean(), col_data.std(), col_data.max() - col_data.min()]
    for row in range(min(10, m.shape[0])):
        row_data = m[row, :]
        features += [row_data.mean(), row_data.std()]
    return np.array(features, dtype=np.float32)

def build_feature_dataset(data_list, labels, thicknom_list):
    features = [extract_features(matrix, thicknom) for matrix, thicknom in zip(data_list, thicknom_list)]
    return np.array(features), np.array(labels)


In [ ]:
def extract_cnn_embeddings(model, loader, device):
    model.eval()
    embeddings, labels = [], []
    with torch.no_grad():
        for data, lbl, _, _ in loader:
            data = data.to(device)
            emb = model(data)
            embeddings.append(emb.cpu().numpy())
            labels.extend(lbl.numpy())
    return np.vstack(embeddings), np.array(labels)


In [ ]:
# LOAD DATA - CHANGE DATA_FOLDER TO YOUR PATH
DATA_FOLDER = 'data/raw'
print('Loading data...')
data_list, labels, thicknom_list = load_dataset_from_folder(DATA_FOLDER)
X_train, X_val, y_train, y_val, thicknom_train, thicknom_val = train_test_split(
    data_list, labels, thicknom_list, test_size=0.2, random_state=42, stratify=labels
)
print(f'Train: {len(X_train)}, Val: {len(X_val)}')
BATCH_SIZE = 32
train_dataset = PipeDataset(X_train, y_train, scaler=None, fit_scaler=True)
val_dataset = PipeDataset(X_val, y_val, scaler=train_dataset.scaler, fit_scaler=False)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')


In [ ]:
# TRAIN CNN FOR EMBEDDINGS
class CNNWithClassifier(nn.Module):
    def __init__(self, embedding_dim=64, n_classes=2):
        super().__init__()
        self.encoder = PipeCNNEncoder(embedding_dim=embedding_dim)
        self.classifier = nn.Sequential(nn.Linear(embedding_dim, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, n_classes))
    def forward(self, x):
        return self.classifier(self.encoder(x))

model_full = CNNWithClassifier().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_full.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
EPOCHS = 30
best_val_acc = 0
print('Training CNN...')
for epoch in range(EPOCHS):
    model_full.train()
    correct, total = 0, 0
    for data, lbl, _, _ in train_loader:
        data, lbl = data.to(DEVICE), lbl.to(DEVICE)
        optimizer.zero_grad()
        outputs = model_full(data)
        loss = criterion(outputs, lbl)
        loss.backward()
        optimizer.step()
        _, predicted = outputs.max(1)
        total += lbl.size(0)
        correct += predicted.eq(lbl).sum().item()
    model_full.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for data, lbl, _, _ in val_loader:
            data, lbl = data.to(DEVICE), lbl.to(DEVICE)
            outputs = model_full(data)
            _, predicted = outputs.max(1)
            val_total += lbl.size(0)
            val_correct += predicted.eq(lbl).sum().item()
    val_acc = 100. * val_correct / val_total
    scheduler.step(val_acc)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model_full.state_dict(), 'best_cnn.pth')
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} - Train Acc: {100.*correct/total:.2f}%, Val Acc: {val_acc:.2f}%')
print(f'Done! Best Val Acc: {best_val_acc:.2f}%')
model_full.load_state_dict(torch.load('best_cnn.pth'))
encoder_model = model_full.encoder


In [ ]:
# EXTRACT EMBEDDINGS + MANUAL FEATURES
print('Extracting CNN embeddings...')
X_train_cnn, y_train = extract_cnn_embeddings(encoder_model, train_loader, DEVICE)
X_val_cnn, y_val = extract_cnn_embeddings(encoder_model, val_loader, DEVICE)
print(f'Train CNN: {X_train_cnn.shape}, Val CNN: {X_val_cnn.shape}')
print('Building manual features...')
X_train_feat, _ = build_feature_dataset(X_train, y_train, thicknom_train)
X_val_feat, _ = build_feature_dataset(X_val, y_val, thicknom_val)
print(f'Train Manual: {X_train_feat.shape}, Val Manual: {X_val_feat.shape}')
print('Concatenating features...')
X_train_combined = np.hstack([X_train_cnn, X_train_feat])
X_val_combined = np.hstack([X_val_cnn, X_val_feat])
print(f'Train Combined: {X_train_combined.shape}, Val Combined: {X_val_combined.shape}')


In [ ]:
# TRAIN XGBOOST
print('Training XGBoost...')
scale_pos_weight = (len(y_train) - sum(y_train)) / max(sum(y_train), 1)
xgb_model = XGBClassifier(n_estimators=400, max_depth=8, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42, use_label_encoder=False)
xgb_model.fit(X_train_combined, y_train, eval_set=[(X_val_combined, y_val)], verbose=True)
y_pred = xgb_model.predict(X_val_combined)
y_proba = xgb_model.predict_proba(X_val_combined)[:, 1]
print('\n' + '='*50)
print('MODEL EVALUATION')
print('='*50)
print(classification_report(y_val, y_pred, target_names=['Good', 'Bad']))
print(f'ROC-AUC: {roc_auc_score(y_val, y_proba):.4f}')
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_val, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()
